In [ ]:
# ══════════════════════════════════════════════════════════════
# M3_F01 — VALIDATION  |  Cellule 1 : Montage Drive + Install
# ══════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

!pip install -q flask flask-cors

import os, sys
from pathlib import Path

CODEBASE = Path("/content/drive/MyDrive/EXODUS_V2/03_MODE_ASCENSION/F01_VALIDATION/CODEBASE")
sys.path.insert(0, str(CODEBASE))
print("Drive monté — CODEBASE :", CODEBASE)

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cellule 2 : Init Dossiers + Vérification Inputs
# ══════════════════════════════════════════════════════════════
DRIVE_ROOT  = Path("/content/drive/MyDrive/EXODUS_V3/M3")
SHARED_DIR  = DRIVE_ROOT / "SHARED"
AVATAR_PATH = SHARED_DIR / "avatar.glb"
AUDIO_PATH  = SHARED_DIR / "audio.mp3"

# Auto-création structure Drive au premier run
for folder in [
    SHARED_DIR,
    DRIVE_ROOT / "F01_VALIDATION" / "OUT_REPORT",
    DRIVE_ROOT / "F02_LOGISTICS" / "OUT",
    DRIVE_ROOT / "F03_SCENOGRAPHY" / "OUT",
    DRIVE_ROOT / "F04_PHOTOGRAPHY" / "OUT",
    DRIVE_ROOT / "F05_ALCHEMIST" / "OUT",
    DRIVE_ROOT / "F05_ALCHEMIST" / "OUT_FRAMES",
    DRIVE_ROOT / "F06_CARRIER" / "OUT",
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Structure Drive initialisée :")
print(f"  {SHARED_DIR}")
print()

# Vérification inputs
avatar_ok = AVATAR_PATH.exists()
audio_ok  = AUDIO_PATH.exists()

for label, path in [("avatar.glb", AVATAR_PATH), ("audio.mp3", AUDIO_PATH)]:
    if path.exists():
        size = path.stat().st_size
        print(f"  {label:20s} OK  ({size/1024:.0f} KB)")
    else:
        print(f"  {label:20s} ABSENT")

print()
if not avatar_ok:
    print("ACTION REQUISE : uploader avatar.glb sur Google Drive ici :")
    print(f"  {AVATAR_PATH}")
    print("  → Aller sur drive.google.com et uploader dans EXODUS_V3/M3/SHARED/")
    raise SystemExit("avatar.glb manquant — arrêt")
else:
    if not audio_ok:
        print("Inputs OK (sans audio — run silencieux)")
    else:
        print("Inputs OK — prêt pour la validation")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cellule 3 : Lancement Flask
# ══════════════════════════════════════════════════════════════
import subprocess, threading, time
from google.colab.output import eval_js

# Copier les fichiers codebase dans /content pour la session
import shutil
LOCAL = Path("/content/m3_f01")
LOCAL.mkdir(exist_ok=True)
for f in CODEBASE.glob("*"):
    shutil.copy(f, LOCAL / f.name)

PORT = 5001

def run_flask():
    os.chdir(str(LOCAL))
    os.system(f"python m3_f01_flask.py")

t = threading.Thread(target=run_flask, daemon=True)
t.start()
time.sleep(2)

# URL publique via colab
url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
print(f"
Moniteur F01 disponible :
{url}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cellule 4 : Mode Headless (optionnel)
# Utiliser si on veut passer F01 sans interface graphique
# ══════════════════════════════════════════════════════════════
# import json
# from pathlib import Path
# 
# DRIVE_ROOT  = Path("/content/drive/MyDrive/EXODUS_V3/M3")
# AVATAR_PATH = DRIVE_ROOT / "SHARED" / "avatar.glb"
# AUDIO_PATH  = DRIVE_ROOT / "SHARED" / "audio.mp3"
# OUT_DIR     = DRIVE_ROOT / "F01_VALIDATION" / "OUT_REPORT"
# 
# # Lire duration audio via pydub (si dispo) ou supposer
# # audio_duration = ... (à remplir manuellement ou via pydub)
# 
# report = {
#     "status": "OK",
#     "has_audio": AUDIO_PATH.exists(),
#     "audio_duration": 0.0,  # à remplir
#     "anim_duration": 0.0,   # à remplir
#     "selected_clip": "Dance_01",
#     "all_clips": [],
#     "margin_s": 0.0
# }
# OUT_DIR.mkdir(parents=True, exist_ok=True)
# with open(OUT_DIR / "m3_f01_report.json", "w") as f:
#     json.dump(report, f, indent=2)
# print("Rapport sauvé :", OUT_DIR / "m3_f01_report.json")